# Project workflow

This notebook demonstrates the end-to-end pipeline for turning a PDF into searchable embeddings:

1. Extract semantic elements from the PDF with `unstructured`
2. Chunk the document by headings and section boundaries
3. Normalize chunks into a stable schema
4. Generate vector embeddings
5. Store the chunks in ChromaDB
6. Run a semantic search query

The sample input lives in `Training_Data/` and the vector store is written to `./chroma_db`.


In [1]:
# Optional: confirm that Tesseract is available on this system.
# This is useful when OCR is needed for scanned pages in the PDF.
# import subprocess
# result = subprocess.run(["which", "tesseract"], capture_output=True, text=True)
# print(result.stdout)  # shows the actual path

# 1. Basic PDF Preprocessing with Unstructured

This step extracts structured semantic elements from the PDF, including headings, paragraphs, lists, and other document blocks.

The `hi_res` strategy is used here because it performs better on formatted documents and helps preserve layout-aware content for downstream retrieval.


In [2]:
import os

# Ensure Homebrew's Tesseract binary is available for OCR on macOS.
# This is optional if OCR is not required for the current PDF.
os.environ["PATH"] = "/opt/homebrew/bin:" + os.environ["PATH"]

In [3]:
SOURCE_FILE_TO_ADD = "Training_Data/AML_NOTES_UNIT_1_2_3_4_5_merged.pdf"

In [4]:
from unstructured.partition.pdf import partition_pdf

# Extract semantic elements from the sample PDF.
# `hi_res` is selected for cleaner layout-aware parsing and better OCR fallback.
elements = partition_pdf(
    filename=SOURCE_FILE_TO_ADD,
    strategy="hi_res",
    languages=["eng"],
    include_metadata=True,
    infer_table_structure=True,
    extract_images_in_pdf=False,
)

/Users/adityabhagwat/Projects/Unstructured-io-Document-Processing-Pipeline/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 367/367 [00:00<00:00, 18192.78it/s]


In [5]:
print(f"Total elements: {len(elements)}")

Total elements: 853


In [6]:
for el in elements[:50]:
    print(type(el))
    print(el.text)
    print(el.metadata)
    print("=" * 80)

<class 'unstructured.documents.elements.Title'>
Unsupervised Learning :
<class 'unstructured.documents.elements.Title'>
What is Unsupervised Learning?
<class 'unstructured.documents.elements.NarrativeText'>
As the name suggests, unsupervised learning is a machine learning technique in which models are not supervised using training dataset. Instead, models itself find the hidden patterns and insights from the given data. It can be compared to learning which takes place in the human brain while learning new things. It can be defined as:
<class 'unstructured.documents.elements.NarrativeText'>
Unsupervised learning is a type of machine learning in which models are trained using unlabelled dataset and are allowed to act on that data without any supervision.
<class 'unstructured.documents.elements.NarrativeText'>
Unsupervised learning cannot be directly applied to a regression or classification problem because unlike supervised learning, we have the input data but no corresponding output dat

# 2. Chunk the Document

Chunking happens after extraction so the model receives semantically meaningful segments rather than large, unstructured blobs of text.

Using `chunk_by_title()` helps keep sections together, preserves headings, and reduces the risk of mixing unrelated topics in a single chunk.


Why chunk_by_title() is better:

* respects headings
* preserves sections
* prevents chunk mixing across topics
* ideal for PDFs

In [7]:
from unstructured.chunking.title import chunk_by_title

# Split the extracted content into section-aware chunks.
# This keeps headings and related paragraphs together, which improves retrieval quality.
chunks = chunk_by_title(
    elements,
    max_characters=1200,
    new_after_n_chars=1000,
    combine_text_under_n_chars=200,
)

In [8]:
print(f"Total chunks: {len(chunks)}")

Total chunks: 140


In [9]:
for chunk in chunks[:3]:
    print(chunk.text)
    print("=" * 80)

Unsupervised Learning :

What is Unsupervised Learning?

As the name suggests, unsupervised learning is a machine learning technique in which models are not supervised using training dataset. Instead, models itself find the hidden patterns and insights from the given data. It can be compared to learning which takes place in the human brain while learning new things. It can be defined as:

Unsupervised learning is a type of machine learning in which models are trained using unlabelled dataset and are allowed to act on that data without any supervision.

Unsupervised learning cannot be directly applied to a regression or classification problem because unlike supervised learning, we have the input data but no corresponding output data. The goal of unsupervised learning is to find the underlying structure of dataset, group that data according to similarities, and represent that dataset in a compressed format.
Example: Suppose the unsupervised learning algorithm is given an input dataset co

# 3. Normalize the Chunks

After chunking, we normalize each chunk into a plain dictionary with a fixed schema.
This decouples all downstream steps — embeddings, ChromaDB storage, and retrieval —
from `unstructured`'s internal object types.

From this point on, the pipeline only works with plain dicts.

In [10]:
normalized_chunks = []

for i, chunk in enumerate(chunks):
    normalized_chunks.append({
        "id":           f"chunk_{i}",
        "text":         chunk.text.strip(),
        "element_type": chunk.category,
        "page_number":  getattr(chunk.metadata, "page_number", None) or 0,
        "section":      getattr(chunk.metadata, "section", None) or
                        getattr(chunk.metadata, "title",   None) or "",
    })

print(normalized_chunks[0])

{'id': 'chunk_0', 'text': 'Unsupervised Learning :\n\nWhat is Unsupervised Learning?\n\nAs the name suggests, unsupervised learning is a machine learning technique in which models are not supervised using training dataset. Instead, models itself find the hidden patterns and insights from the given data. It can be compared to learning which takes place in the human brain while learning new things. It can be defined as:\n\nUnsupervised learning is a type of machine learning in which models are trained using unlabelled dataset and are allowed to act on that data without any supervision.\n\nUnsupervised learning cannot be directly applied to a regression or classification problem because unlike supervised learning, we have the input data but no corresponding output data. The goal of unsupervised learning is to find the underlying structure of dataset, group that data according to similarities, and represent that dataset in a compressed format.', 'element_type': 'CompositeElement', 'page_nu

# 4. Create Embeddings

This step converts each chunk into a dense numeric representation so the text can be searched semantically.

`all-MiniLM-L6-v2` is a lightweight sentence-transformer model that works well for local experimentation and fast retrieval pipelines.


In [11]:
from sentence_transformers import SentenceTransformer

# Use a compact sentence-transformer model for local embedding generation.
# It is a good default for experimentation and retrieval pipelines.
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

chunk_texts = [c["text"] for c in normalized_chunks]
embeddings = embedding_model.encode(chunk_texts, show_progress_bar=True)

Batches: 100%|██████████| 5/5 [00:00<00:00,  5.76it/s]


# Bonus Steps

# 5. Insert Embeddings into ChromaDB

This section writes the chunk embeddings and their metadata into a persistent ChromaDB collection so the data can be reused across sessions.


In [12]:
import uuid
import chromadb
from datetime import datetime, timezone

SOURCE_FILE = SOURCE_FILE_TO_ADD
DOCUMENT_ID = str(uuid.uuid5(uuid.NAMESPACE_URL, SOURCE_FILE))
created_at  = datetime.now(timezone.utc).isoformat()

client     = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="single_pdf_collection")

In [13]:
for i, chunk in enumerate(normalized_chunks):
    collection.add(
        ids        = [str(uuid.uuid4())],
        documents  = [chunk["text"]],
        embeddings = [embeddings[i].tolist()],
        metadatas  = [{
            "document_id":  DOCUMENT_ID,
            "chunk_id":     chunk["id"],
            "element_type": chunk["element_type"],
            "page_number":  chunk["page_number"],
            "section":      chunk["section"],
            "source_file":  SOURCE_FILE,
            "created_at":   created_at,
        }]
    )

print(f"Inserted {len(normalized_chunks)} chunks into ChromaDB")

Inserted 140 chunks into ChromaDB


# 6. Query in ChromaDB

The final step embeds the user query and searches the vector store for the most semantically similar chunks.

This makes it possible to retrieve context that matches the intent of the query, even if the wording differs from the source document.


In [14]:
query = "What is k means clustering?"

# Embed the search query and retrieve the closest matching chunks from ChromaDB.
query_embedding = embedding_model.encode([query])[0]

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

In [15]:
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(f"[page {meta['page_number']} | {meta['section']} | {meta['element_type']}]")
    print(doc)
    print("=" * 80)

[page 4 |  | CompositeElement]
o Determines the best value for K centre points or centroids by an iterative process.

o Assigns each data point to its closest k-centre. Those data points which are near to the particular k-centre, create a cluster.

Hence each cluster has datapoints with some commonalities, and it is away from other clusters.

The below diagram explains the working of the K-means Clustering Algorithm:

Before K-Means After K-Means
[page 3 |  | CompositeElement]
Unsupervised Learning algorithms:

Below is the list of some popular unsupervised learning algorithms:

o K-means clustering

o Hierarchal clustering

K-Means Clustering Algorithm

K-Means Clustering is an Unsupervised Learning algorithm, which groups the unlabelled dataset into different clusters. Here K defines the number of pre- defined clusters that need to be created in the process, as if K=2, there will be two clusters, and for K=3, there will be three clusters, and so on.

It is an iterative algorithm that